# B_S3.5 - Detection-Rate Heatmaps by Relationship Type

Goal: visualize which metrics detect which relationship types under the same alpha-calibrated max-Z logic used in `B_S3.4`.

This notebook does **not** compare raw metric values across metrics. Raw values are on incompatible scales. Instead each cell is a validation detection rate:

```text
cell = fraction of validation cases in this row detected by this metric or metric combo
```

For `true_null`, the cell is the empirical null detected rate. Ideally it is close to `alpha = 0.05`.

For signal rows, the cell is detection power.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.grid': False,
})

# Make paths work whether the notebook is run from repo root or from B/.
ROOT = Path.cwd()
if not (ROOT / 'output' / 'S3' / 'permutation_all.parquet').exists() and (ROOT / 'B' / 'output' / 'S3' / 'permutation_all.parquet').exists():
    ROOT = ROOT / 'B'

S1_DIR = ROOT / 'output' / 'S1'
S3_DIR = ROOT / 'output' / 'S3'
OUT_DIR = ROOT / 'output' / 'S3.5'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260623
ALPHA = 0.05
TRAIN_FRACTION = 0.5

EXCLUDE_BANDWIDTH = True
MERGE_DCOR_DCOV = True

print(f'ROOT = {ROOT}')
print(f'OUT_DIR = {OUT_DIR}')

## 1. Load S3 Data and Recreate S3.4 Split

This uses the same MINE-covered subset and stratified train/validation split as S3.4, so the heatmap values are directly comparable with S3.4 selection results.

In [ ]:
df = pd.read_parquet(S3_DIR / 'permutation_all.parquet')
Z_COLS = sorted([c for c in df.columns if c.startswith('z_')])
P_COLS = sorted([c for c in df.columns if c.startswith('p_') and c != 'p_value'])

cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_null['case_id'] = cases_null['case_id'] + cases_main['case_id'].max()

meta_cols = [
    'case_id', 'family_id', 'family_name', 'snr', 'spread_pattern',
    'x_distribution', 'equation_presentation', 'basic_label'
]
for col in meta_cols:
    if col not in cases_main.columns:
        cases_main[col] = np.nan
    if col not in cases_null.columns:
        cases_null[col] = np.nan

cases_all = pd.concat([cases_main[meta_cols], cases_null[meta_cols]], ignore_index=True)
df = df.merge(cases_all, on='case_id', how='left')

is_null_family = df['family_id'] == 'Null'
is_constant_spread = df['spread_pattern'] == 'constant'

df['category'] = 'mean+variance'
df.loc[is_null_family & is_constant_spread, 'category'] = 'true_null'
df.loc[~is_null_family & is_constant_spread, 'category'] = 'mean_only'
df.loc[is_null_family & ~is_constant_spread, 'category'] = 'variance_only'

print(f'Before MINE coverage filter: {len(df):,} cases x {len(Z_COLS)} Z metrics')

mine_z_cols = [c for c in Z_COLS if c in ('z_mic', 'z_mas', 'z_mev', 'z_mcn')]
mine_coverage_mask = df[mine_z_cols].notna().all(axis=1)
df = df[mine_coverage_mask].reset_index(drop=True)

assert not any(df[zc].isna().any() for zc in Z_COLS), 'Some Z columns still contain NaN after MINE filter.'
print(f'After MINE coverage filter: {len(df):,} cases')
print(df['category'].value_counts().to_string())

rng = np.random.default_rng(RANDOM_SEED)
df['split'] = 'validation'
for _, idx in df.groupby(['category', 'x_distribution'], dropna=False).groups.items():
    idx = np.array(list(idx)).copy()
    rng.shuffle(idx)
    n_train = int(round(len(idx) * TRAIN_FRACTION))
    if len(idx) > 1:
        n_train = min(max(n_train, 1), len(idx) - 1)
    df.loc[idx[:n_train], 'split'] = 'train'

train = df[df['split'] == 'train'].copy()
validation = df[df['split'] == 'validation'].copy()

split_counts = df.groupby(['split', 'category']).size().unstack(fill_value=0)
print('\nSplit counts:')
print(split_counts.to_string())

## 2. Metric Names, Groups, and Safe Metric Set

The safe metric set follows S3.4: remove degenerate/unstable metrics, optionally remove raw bandwidth metrics, and keep `dcor` rather than both `dcor` and `dcov`.

In [ ]:
def metric_display(col):
    name = col.replace('z_', '')
    replacements = {
        'abs_pearson_r': '|pearson|',
        'abs_spearman_rho': '|spearman|',
        'abs_covariance': '|covariance|',
        'mic_minus_r2': 'MIC-r2',
    }
    return replacements.get(name, name)

METRIC_DISPLAY = {c: metric_display(c) for c in Z_COLS}

METRIC_GROUPS = {}
for c in Z_COLS:
    name = c.replace('z_', '')
    if name in ('pearson', 'spearman', 'abs_pearson_r', 'abs_spearman_rho', 'abs_covariance'):
        METRIC_GROUPS[c] = 'correlation'
    elif 'dcor' in name or 'dcov' in name:
        METRIC_GROUPS[c] = 'distance'
    elif 'ep_' in name or 'pf_' in name or 'seg_' in name:
        METRIC_GROUPS[c] = 'slope'
    elif 'bin_' in name or name == 'eta2':
        METRIC_GROUPS[c] = 'bin'
    elif 'dist_' in name:
        METRIC_GROUPS[c] = 'distribution'
    elif name in ('mic', 'mas', 'mev', 'mcn', 'mic_minus_r2'):
        METRIC_GROUPS[c] = 'mine'
    elif name == 'lowess_r2':
        METRIC_GROUPS[c] = 'nonlinear'
    else:
        METRIC_GROUPS[c] = 'other'

train_null = train[train['category'] == 'true_null']
calib_rows = []
for zc in Z_COLS:
    z_null = train_null[zc].dropna()
    pc = zc.replace('z_', 'p_')
    p_null = train_null[pc].dropna() if pc in train_null.columns else pd.Series(dtype=float)
    empirical_fpr = float((p_null <= ALPHA).mean()) if len(p_null) else np.nan
    calib_rows.append({
        'metric': zc,
        'display': METRIC_DISPLAY[zc],
        'group': METRIC_GROUPS[zc],
        'mean_train_null': float(z_null.mean()) if len(z_null) else np.nan,
        'std_train_null': float(z_null.std()) if len(z_null) > 1 else np.nan,
        'empirical_p_fpr_train_null': empirical_fpr,
    })

calib = pd.DataFrame(calib_rows)
calib['flag_bias'] = calib['mean_train_null'].abs() > 1.0
calib['flag_degen'] = calib['std_train_null'] < 0.3
calib['flag_unstable'] = calib['std_train_null'] > 3.0
calib['flag_high_p_fpr'] = calib['empirical_p_fpr_train_null'] > 0.15
calib['flagged'] = calib[['flag_bias', 'flag_degen', 'flag_unstable', 'flag_high_p_fpr']].any(axis=1)

SAFE_METRICS = calib.loc[~calib['flagged'], 'metric'].tolist()

if EXCLUDE_BANDWIDTH:
    SAFE_METRICS = [m for m in SAFE_METRICS if '_bw_' not in m]

if MERGE_DCOR_DCOV and 'z_dcov' in SAFE_METRICS and 'z_dcor' in SAFE_METRICS:
    SAFE_METRICS = [m for m in SAFE_METRICS if m != 'z_dcov']

print(f'Safe metrics: {len(SAFE_METRICS)} / {len(Z_COLS)}')
print('Flagged metrics:')
print(calib.loc[calib['flagged'], ['display', 'group', 'mean_train_null', 'std_train_null']].to_string(index=False))

## 3. Detection Helpers

For an individual metric, the statistic is its Z-score. For a metric combo, the statistic is `max(Z)` across the combo. The threshold is always fit from train `true_null` only.

In [ ]:
def fit_threshold(metrics, alpha=ALPHA):
    metrics = list(metrics)
    T_null = train.loc[train['category'] == 'true_null', metrics].max(axis=1).dropna()
    return float(np.nanpercentile(T_null, 100 * (1 - alpha)))


def detect_with_metrics(data, metrics, threshold=None):
    metrics = list(metrics)
    if threshold is None:
        threshold = fit_threshold(metrics)
    T = data[metrics].max(axis=1)
    detected = (T > threshold).fillna(False)
    return detected, T, threshold


def detection_table_by_row(data, specs, row_col, row_order=None, min_n=1):
    rows = []
    for spec_name, metrics in specs.items():
        detected, _, threshold = detect_with_metrics(data, metrics)
        tmp = data[[row_col]].copy()
        tmp['detected'] = detected.values
        tmp = tmp.dropna(subset=[row_col])
        grouped = tmp.groupby(row_col)['detected'].agg(['mean', 'size']).reset_index()
        grouped['spec'] = spec_name
        grouped['threshold'] = threshold
        rows.append(grouped)
    long = pd.concat(rows, ignore_index=True)
    long = long[long['size'] >= min_n]
    table = long.pivot(index=row_col, columns='spec', values='mean')
    table = table.reindex(columns=list(specs.keys()))
    counts = long.drop_duplicates(subset=[row_col]).set_index(row_col)['size']
    if row_order is not None:
        keep = [r for r in row_order if r in table.index]
        table = table.loc[keep]
        counts = counts.loc[keep]
    return table, counts, long


def plot_heatmap(table, title, path, counts=None, cmap='YlOrRd', vmin=0.0, vmax=1.0, fmt='.2f'):
    plot_table = table.copy()
    n_rows, n_cols = plot_table.shape
    fig_w = max(8, 0.85 * n_cols + 3.5)
    fig_h = max(3.2, 0.38 * n_rows + 1.8)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    im = ax.imshow(plot_table.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(plot_table.columns, rotation=45, ha='right', fontsize=8)

    ylabels = list(plot_table.index)
    if counts is not None:
        ylabels = [f'{idx} (n={int(counts.loc[idx])})' if idx in counts.index else str(idx) for idx in plot_table.index]
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(ylabels, fontsize=8)

    for i in range(n_rows):
        for j in range(n_cols):
            val = plot_table.iat[i, j]
            if pd.isna(val):
                text = ''
            else:
                text = format(val, fmt)
            color = 'white' if pd.notna(val) and val >= 0.65 else 'black'
            ax.text(j, i, text, ha='center', va='center', fontsize=7, color=color)

    ax.set_title(title, fontsize=11, pad=12)
    cbar = fig.colorbar(im, ax=ax, shrink=0.75)
    cbar.set_label('Detection rate')
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    plt.show()
    print(f'Saved -> {path}')

## 4. Individual Metric Heatmap

These are single-metric detectors. Each metric gets its own train-null 95% threshold.

In [ ]:
preferred_individual = [
    'z_dcor',
    'z_mic', 'z_mev', 'z_mas', 'z_mic_minus_r2',
    'z_ew_dist_ks', 'z_ec_dist_ks', 'z_ew_dist_wass', 'z_ec_dist_wass',
    'z_abs_pearson_r', 'z_abs_spearman_rho',
    'z_ew_bin_eta2', 'z_ec_bin_amp', 'z_lowess_r2',
]
individual_metrics = [m for m in preferred_individual if m in SAFE_METRICS]
individual_specs = {METRIC_DISPLAY[m]: [m] for m in individual_metrics}

category_order = ['true_null', 'mean_only', 'variance_only', 'mean+variance']
ind_cat, ind_cat_counts, ind_cat_long = detection_table_by_row(
    validation, individual_specs, 'category', row_order=category_order
)

ind_cat.to_csv(OUT_DIR / 'individual_detection_by_category.csv', float_format='%.6f')
ind_cat_long.to_csv(OUT_DIR / 'individual_detection_by_category_long.csv', index=False, float_format='%.6f')

plot_heatmap(
    ind_cat,
    'Individual Metric Detection Rate by Category (validation)',
    OUT_DIR / 'fig1_individual_detection_by_category.png',
    counts=ind_cat_counts,
)

ind_cat

## 5. Individual Metric Heatmap by Relationship Family

This version is closer to the example figure: rows are relationship families. Mean-only and mean+variance cases are combined by family because both contain the same mean-response shape. Variance-only rows are shown separately by spread pattern.

In [ ]:
def relation_label(row):
    if row['category'] == 'true_null':
        return 'No relationship'
    if row['category'] == 'variance_only':
        return f'Variance only: {row["spread_pattern"]}'
    return f'{row["family_id"]}: {row["family_name"]}'

validation = validation.copy()
validation['relationship_type'] = validation.apply(relation_label, axis=1)

family_order = ['No relationship']
family_order += [
    'Variance only: increasing',
    'Variance only: decreasing',
    'Variance only: middle_high',
]
family_meta = (
    validation[validation['family_id'].str.startswith('F', na=False)]
    [['family_id', 'family_name']]
    .drop_duplicates()
    .sort_values('family_id')
)
family_order += [f'{r.family_id}: {r.family_name}' for r in family_meta.itertuples(index=False)]

ind_family, ind_family_counts, ind_family_long = detection_table_by_row(
    validation, individual_specs, 'relationship_type', row_order=family_order
)

ind_family.to_csv(OUT_DIR / 'individual_detection_by_relationship_type.csv', float_format='%.6f')
ind_family_long.to_csv(OUT_DIR / 'individual_detection_by_relationship_type_long.csv', index=False, float_format='%.6f')

plot_heatmap(
    ind_family,
    'Individual Metric Detection Rate by Relationship Type (validation)',
    OUT_DIR / 'fig2_individual_detection_by_relationship_type.png',
    counts=ind_family_counts,
)

ind_family.head(12)

## 6. Metric Combo Heatmaps

These show why the final sparse choice can be smaller than the intuitive full set. Each combo gets a fresh train-null threshold for its own `max(Z)` statistic.

In [ ]:
def available(metrics):
    return [m for m in metrics if m in SAFE_METRICS]

combo_specs = {
    'dcor': available(['z_dcor']),
    'MINE only': available(['z_mas', 'z_mev', 'z_mic', 'z_mic_minus_r2']),
    'distribution only': available(['z_ec_dist_ks', 'z_ec_dist_wass', 'z_ew_dist_ks', 'z_ew_dist_wass']),
    'mean-response only': available(['z_abs_pearson_r', 'z_abs_spearman_rho', 'z_ew_bin_eta2', 'z_ec_bin_amp', 'z_lowess_r2']),
    'dcor + mic': available(['z_dcor', 'z_mic']),
    'dcor + ew_wass + mic': available(['z_dcor', 'z_ew_dist_wass', 'z_mic']),
    'all_safe': SAFE_METRICS,
}
combo_specs = {name: metrics for name, metrics in combo_specs.items() if metrics}

combo_cat, combo_cat_counts, combo_cat_long = detection_table_by_row(
    validation, combo_specs, 'category', row_order=category_order
)
combo_family, combo_family_counts, combo_family_long = detection_table_by_row(
    validation, combo_specs, 'relationship_type', row_order=family_order
)

combo_cat.to_csv(OUT_DIR / 'combo_detection_by_category.csv', float_format='%.6f')
combo_cat_long.to_csv(OUT_DIR / 'combo_detection_by_category_long.csv', index=False, float_format='%.6f')
combo_family.to_csv(OUT_DIR / 'combo_detection_by_relationship_type.csv', float_format='%.6f')
combo_family_long.to_csv(OUT_DIR / 'combo_detection_by_relationship_type_long.csv', index=False, float_format='%.6f')

plot_heatmap(
    combo_cat,
    'Metric Combo Detection Rate by Category (validation)',
    OUT_DIR / 'fig3_combo_detection_by_category.png',
    counts=combo_cat_counts,
)

plot_heatmap(
    combo_family,
    'Metric Combo Detection Rate by Relationship Type (validation)',
    OUT_DIR / 'fig4_combo_detection_by_relationship_type.png',
    counts=combo_family_counts,
)

combo_cat

## 7. Low-SNR Diagnostic Heatmap

The selection results are already near ceiling for many relationship types. Low SNR is where metrics differ most, so this table repeats the combo heatmap for the lowest finite SNR levels. In this synthetic design, `true_null` and `variance_only` do not have finite SNR values, so this diagnostic focuses on mean-containing signal rows.

In [ ]:
finite_snr = sorted([s for s in validation['snr'].dropna().unique() if np.isfinite(s)])
low_snr_levels = finite_snr[:4]
low_snr = validation[validation['snr'].isin(low_snr_levels)].copy()

print(f'Low SNR levels: {low_snr_levels}')
print(low_snr.groupby(['snr', 'category']).size().unstack(fill_value=0).to_string())

low_combo_cat, low_combo_cat_counts, low_combo_cat_long = detection_table_by_row(
    low_snr, combo_specs, 'category', row_order=category_order
)
low_combo_cat.to_csv(OUT_DIR / 'combo_detection_by_category_low_snr.csv', float_format='%.6f')
low_combo_cat_long.to_csv(OUT_DIR / 'combo_detection_by_category_low_snr_long.csv', index=False, float_format='%.6f')

plot_heatmap(
    low_combo_cat,
    'Metric Combo Detection Rate by Category, Finite Low-SNR Mean Cases Only (validation)',
    OUT_DIR / 'fig5_combo_detection_by_category_low_snr.png',
    counts=low_combo_cat_counts,
)

low_combo_cat

## 8. Metric Score Landscapes

These plots are closer to the MIC paper-style figure, but adapted to this workflow. They show metric-specific permutation Z-scores against relationship strength, with the train-null alpha threshold drawn in each panel.

There are two separate landscape views:

```text
Mean-containing cases: x = SNR / (1 + SNR)
Variance-only cases:   x = spread pattern
```

They are separated because `variance_only` cases do not have finite SNR in this synthetic design.

In [ ]:
landscape_metrics = [
    'z_dcor',
    'z_mic', 'z_mev', 'z_mas', 'z_mic_minus_r2',
    'z_ew_dist_ks', 'z_ec_dist_ks', 'z_ew_dist_wass', 'z_ec_dist_wass',
    'z_abs_pearson_r', 'z_abs_spearman_rho',
    'z_ew_bin_eta2', 'z_ec_bin_amp', 'z_lowess_r2',
]
landscape_metrics = [m for m in landscape_metrics if m in SAFE_METRICS]

thresholds = {m: fit_threshold([m]) for m in landscape_metrics}
metric_groups = {m: METRIC_GROUPS[m] for m in landscape_metrics}

print('Landscape metrics:')
for m in landscape_metrics:
    print(f'  {METRIC_DISPLAY[m]:16s} group={METRIC_GROUPS[m]:12s} threshold={thresholds[m]:.3f}')

### 8.1 Mean-Containing Landscape by SNR

This plot includes `true_null` cases at `x = 0` as a reference band, then mean-containing signal cases by signal strength.

In [ ]:
mean_landscape = validation[
    (validation['category'].isin(['true_null', 'mean_only', 'mean+variance']))
].copy()

mean_landscape['signal_strength'] = 0.0
finite_signal = mean_landscape['category'].isin(['mean_only', 'mean+variance']) & np.isfinite(mean_landscape['snr'])
mean_landscape.loc[finite_signal, 'signal_strength'] = (
    mean_landscape.loc[finite_signal, 'snr'] / (1.0 + mean_landscape.loc[finite_signal, 'snr'])
)
mean_landscape = mean_landscape[mean_landscape['category'].eq('true_null') | finite_signal].copy()

# Jitter true_null x positions so the null band is visible.
rng = np.random.default_rng(RANDOM_SEED)
mean_landscape['x_plot'] = mean_landscape['signal_strength']
null_mask = mean_landscape['category'].eq('true_null')
mean_landscape.loc[null_mask, 'x_plot'] = rng.uniform(-0.012, 0.012, null_mask.sum())

mean_long_rows = []
for m in landscape_metrics:
    tmp = mean_landscape[['case_id', 'category', 'family_id', 'family_name', 'snr', 'signal_strength', 'x_plot', m]].copy()
    tmp = tmp.rename(columns={m: 'z_score'})
    tmp['metric'] = m
    tmp['display'] = METRIC_DISPLAY[m]
    tmp['threshold'] = thresholds[m]
    tmp['detected'] = tmp['z_score'] > tmp['threshold']
    mean_long_rows.append(tmp)
mean_landscape_long = pd.concat(mean_long_rows, ignore_index=True)
mean_landscape_long.to_csv(OUT_DIR / 'metric_z_landscape_mean_cases.csv', index=False, float_format='%.6f')

n_cols = 4
n_rows = int(np.ceil(len(landscape_metrics) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.0, n_rows * 3.0), sharex=True)
axes_flat = np.atleast_1d(axes).flat

cat_colors = {
    'true_null': '#9e9e9e',
    'mean_only': '#3b82c4',
    'mean+variance': '#2ca25f',
}

for ax, m in zip(axes_flat, landscape_metrics):
    sub = mean_landscape_long[mean_landscape_long['metric'].eq(m)].dropna(subset=['z_score'])
    for cat in ['true_null', 'mean_only', 'mean+variance']:
        s = sub[sub['category'].eq(cat)]
        if len(s) == 0:
            continue
        alpha = 0.20 if cat == 'true_null' else 0.35
        size = 7 if cat == 'true_null' else 10
        ax.scatter(s['x_plot'], s['z_score'], s=size, alpha=alpha,
                   color=cat_colors[cat], edgecolors='none', label=cat)

    thr = thresholds[m]
    ax.axhline(thr, color='black', lw=1.0, ls='--')
    q99 = np.nanpercentile(sub['z_score'], 99.2)
    q01 = np.nanpercentile(sub['z_score'], 0.5)
    upper = max(thr + 0.5, q99 * 1.05)
    lower = min(-0.5, q01)
    ax.set_ylim(lower, upper)
    ax.set_xlim(-0.04, 1.02)
    ax.set_title(f'{METRIC_DISPLAY[m]}\n{METRIC_GROUPS[m]}, thr={thr:.2f}', fontsize=9)
    ax.grid(True, alpha=0.25)

for ax in axes_flat[len(landscape_metrics):]:
    ax.set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.01))
fig.supxlabel('Signal strength = SNR / (1 + SNR); true_null shown near 0')
fig.supylabel('Permutation Z-score')
fig.suptitle('Metric Z-Score Landscapes for Mean-Containing Relationships', fontsize=13, y=1.04)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig6_metric_z_landscapes_mean_cases.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUT_DIR / "fig6_metric_z_landscapes_mean_cases.png"}')

### 8.2 Variance-Only Landscape by Spread Pattern

Variance-only cases have constant mean and no finite SNR. This view shows whether each metric responds to changing spread while keeping `true_null` as the left reference group.

In [ ]:
var_landscape = validation[validation['category'].isin(['true_null', 'variance_only'])].copy()
spread_order = ['true_null', 'increasing', 'decreasing', 'middle_high']
var_landscape['spread_axis'] = np.where(
    var_landscape['category'].eq('true_null'),
    'true_null',
    var_landscape['spread_pattern'].astype(str),
)
var_landscape = var_landscape[var_landscape['spread_axis'].isin(spread_order)].copy()
axis_map = {name: i for i, name in enumerate(spread_order)}
var_landscape['x_base'] = var_landscape['spread_axis'].map(axis_map).astype(float)
rng = np.random.default_rng(RANDOM_SEED + 1)
var_landscape['x_plot'] = var_landscape['x_base'] + rng.uniform(-0.12, 0.12, len(var_landscape))

var_long_rows = []
for m in landscape_metrics:
    tmp = var_landscape[['case_id', 'category', 'spread_pattern', 'spread_axis', 'x_plot', m]].copy()
    tmp = tmp.rename(columns={m: 'z_score'})
    tmp['metric'] = m
    tmp['display'] = METRIC_DISPLAY[m]
    tmp['threshold'] = thresholds[m]
    tmp['detected'] = tmp['z_score'] > tmp['threshold']
    var_long_rows.append(tmp)
var_landscape_long = pd.concat(var_long_rows, ignore_index=True)
var_landscape_long.to_csv(OUT_DIR / 'metric_z_landscape_variance_only.csv', index=False, float_format='%.6f')

n_cols = 4
n_rows = int(np.ceil(len(landscape_metrics) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.0, n_rows * 3.0), sharex=True)
axes_flat = np.atleast_1d(axes).flat

spread_colors = {
    'true_null': '#9e9e9e',
    'increasing': '#d95f02',
    'decreasing': '#7570b3',
    'middle_high': '#1b9e77',
}

for ax, m in zip(axes_flat, landscape_metrics):
    sub = var_landscape_long[var_landscape_long['metric'].eq(m)].dropna(subset=['z_score'])
    for spread in spread_order:
        s = sub[sub['spread_axis'].eq(spread)]
        if len(s) == 0:
            continue
        alpha = 0.20 if spread == 'true_null' else 0.35
        size = 7 if spread == 'true_null' else 10
        ax.scatter(s['x_plot'], s['z_score'], s=size, alpha=alpha,
                   color=spread_colors[spread], edgecolors='none', label=spread)

    thr = thresholds[m]
    ax.axhline(thr, color='black', lw=1.0, ls='--')
    q99 = np.nanpercentile(sub['z_score'], 99.2)
    q01 = np.nanpercentile(sub['z_score'], 0.5)
    upper = max(thr + 0.5, q99 * 1.05)
    lower = min(-0.5, q01)
    ax.set_ylim(lower, upper)
    ax.set_xlim(-0.45, len(spread_order) - 0.55)
    ax.set_title(f'{METRIC_DISPLAY[m]}\n{METRIC_GROUPS[m]}, thr={thr:.2f}', fontsize=9)
    ax.grid(True, axis='y', alpha=0.25)
    ax.set_xticks(range(len(spread_order)))
    ax.set_xticklabels(['null', 'incr', 'decr', 'middle'], rotation=0, fontsize=8)

for ax in axes_flat[len(landscape_metrics):]:
    ax.set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, 1.01))
fig.supxlabel('Spread pattern')
fig.supylabel('Permutation Z-score')
fig.suptitle('Metric Z-Score Landscapes for Variance-Only Relationships', fontsize=13, y=1.04)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig7_metric_z_landscapes_variance_only.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUT_DIR / "fig7_metric_z_landscapes_variance_only.png"}')

### 8.3 Aggregated Detection-Rate Landscapes

The raw Z-score scatter landscapes above are dense and mainly useful for diagnostics. For interpretation, these aggregated heatmaps are clearer: each cell is a detection rate after applying the metric-specific train-null alpha threshold.

In [ ]:
def metric_detection_rate(data, metric, mask):
    vals = data.loc[mask, metric].dropna()
    if len(vals) == 0:
        return np.nan
    return float((vals > thresholds[metric]).mean())

snr_levels = sorted([s for s in validation['snr'].dropna().unique() if np.isfinite(s)])

mean_snr_tables = {}
for cat in ['mean_only', 'mean+variance']:
    mat = []
    for metric in landscape_metrics:
        row = []
        for snr in snr_levels:
            mask = validation['category'].eq(cat) & validation['snr'].eq(snr)
            row.append(metric_detection_rate(validation, metric, mask))
        mat.append(row)
    mean_snr_tables[cat] = pd.DataFrame(
        mat,
        index=[METRIC_DISPLAY[m] for m in landscape_metrics],
        columns=[f'{s:g}' for s in snr_levels],
    )
    mean_snr_tables[cat].to_csv(OUT_DIR / f'metric_detection_rate_by_snr_{cat}.csv', float_format='%.6f')

fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.8), sharey=True)
im = None
for ax, cat in zip(axes, ['mean_only', 'mean+variance']):
    table = mean_snr_tables[cat]
    values = table.values.astype(float)
    im = ax.imshow(values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title(cat, fontsize=13)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=8)
    ax.set_xlabel('SNR')
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val = values[i, j]
            if np.isfinite(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6,
                        color='white' if val >= 0.65 else 'black')

axes[0].set_ylabel('Metric')
fig.subplots_adjust(right=0.90, top=0.86, bottom=0.15, wspace=0.08)
cax = fig.add_axes([0.92, 0.20, 0.015, 0.58])
fig.colorbar(im, cax=cax, label='Detection rate')
fig.suptitle('Metric Detection Rate by SNR (validation; individual alpha-calibrated thresholds)', fontsize=14)
fig.savefig(OUT_DIR / 'fig8_metric_detection_rate_by_snr.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUT_DIR / "fig8_metric_detection_rate_by_snr.png"}')

Variance-only cases do not have finite SNR. This heatmap therefore uses spread pattern as the x-axis, with `true_null` included as the reference column.

In [ ]:
spread_cols = ['true_null', 'increasing', 'decreasing', 'middle_high']
spread_rows = []
for metric in landscape_metrics:
    row = []
    row.append(metric_detection_rate(validation, metric, validation['category'].eq('true_null')))
    for spread in ['increasing', 'decreasing', 'middle_high']:
        mask = validation['category'].eq('variance_only') & validation['spread_pattern'].eq(spread)
        row.append(metric_detection_rate(validation, metric, mask))
    spread_rows.append(row)

spread_table = pd.DataFrame(
    spread_rows,
    index=[METRIC_DISPLAY[m] for m in landscape_metrics],
    columns=spread_cols,
)
spread_table.to_csv(OUT_DIR / 'metric_detection_rate_by_spread_pattern.csv', float_format='%.6f')

fig, ax = plt.subplots(figsize=(7.8, 6.2))
values = spread_table.values.astype(float)
im = ax.imshow(values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax.set_title('Metric Detection Rate for Variance-Only Patterns (validation)')
ax.set_xticks(range(len(spread_table.columns)))
ax.set_xticklabels(spread_table.columns, rotation=25, ha='right')
ax.set_yticks(range(len(spread_table.index)))
ax.set_yticklabels(spread_table.index, fontsize=8)
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        val = values[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if val >= 0.65 else 'black')
fig.colorbar(im, ax=ax, shrink=0.75, label='Detection rate')
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig9_metric_detection_rate_by_spread_pattern.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUT_DIR / "fig9_metric_detection_rate_by_spread_pattern.png"}')
spread_table

## 9. Readout

Use Fig. 1 and Fig. 2 to answer: which individual metric detects which relationship type?

Use Fig. 3 and Fig. 4 to answer: does adding MINE/distribution metrics improve coverage beyond `dcor`?

Important interpretation rule:

```text
A high signal detection rate is useful only if the true_null row stays near alpha.
```

In [ ]:
summary_rows = []
for name, metrics in combo_specs.items():
    detected, _, threshold = detect_with_metrics(validation, metrics)
    row = {'combo': name, 'k': len(metrics), 'threshold': threshold}
    for cat in category_order:
        mask = validation['category'] == cat
        row[cat] = float(detected[mask].mean()) if mask.sum() else np.nan
    row['overall_power'] = float(detected[validation['category'] != 'true_null'].mean())
    row['macro_power'] = float(np.nanmean([row['mean_only'], row['variance_only'], row['mean+variance']]))
    summary_rows.append(row)

combo_summary = pd.DataFrame(summary_rows).sort_values(['macro_power', 'overall_power'], ascending=False)
combo_summary.to_csv(OUT_DIR / 'combo_summary.csv', index=False, float_format='%.6f')
print(combo_summary.to_string(index=False, float_format='{:.4f}'.format))
print(f'\nSaved outputs to {OUT_DIR}')